# Day 044 — Exercise 5: delete_item

**What you'll build:** `delete_item(session, item_id) -> bool` — fetch an item by id, delete it, commit, and return `True`. Return `False` if the id does not exist.

**Why it matters:** `session.delete(obj)` marks the object for deletion. The DELETE SQL is generated on the next `commit()`. This is the ORM way to delete — you operate on Python objects, not SQL strings. The bool return lets callers check whether a deletion actually happened without needing to inspect the database again.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def add_item(session, name, category, price, quantity=0):
    item = Item(name=name, category=category, price=price, quantity=quantity)
    session.add(item)
    session.commit()
    session.refresh(item)
    return item


def get_items(session, category=None):
    stmt = select(Item)
    if category is not None:
        stmt = stmt.where(Item.category == category)
    return list(session.execute(stmt).scalars().all())


def update_price(session, item_id, new_price):
    item = session.get(Item, item_id)
    if item is None:
        return None
    item.price = new_price
    session.commit()
    session.refresh(item)
    return item


engine  = setup_engine()
session = Session(engine)

item_a = add_item(session, 'Widget',  'Parts',  5.99, 100)
item_b = add_item(session, 'Gadget',  'Parts', 19.99,  50)
item_c = add_item(session, 'Doohickey','Parts', 3.49, 200)

## Your Implementation

In [ ]:
def delete_item(session, item_id):
    """
    Delete an item by id. Return True if deleted, False if not found.

    Steps:
    1. item = session.get(Item, item_id)
    2. if item is None: return False
    3. session.delete(item)
    4. session.commit()
    5. return True
    """
    # TODO: item = session.get(Item, item_id)
    # TODO: if item is None: return False
    # TODO: session.delete(item)
    # TODO: session.commit()
    # TODO: return True
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'delete_item' in globals()
        passed += 1; print('\u2705 Check 1: delete_item is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns True for existing item
    try:
        result = delete_item(session, item_a.id)
        assert result is True, f'expected True, got {result}'
        passed += 1; print('\u2705 Check 2: returns True for existing item')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: item no longer in DB
    try:
        gone = session.get(Item, item_a.id)
        assert gone is None, \
            f'item should be deleted but found: {gone}'
        passed += 1; print('\u2705 Check 3: item no longer in DB')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: only 2 items remain
    try:
        remaining = get_items(session)
        assert len(remaining) == 2, \
            f'expected 2 remaining, got {len(remaining)}'
        passed += 1; print(f'\u2705 Check 4: {len(remaining)} items remain')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: returns False for non-existent id
    try:
        result2 = delete_item(session, 99999)
        assert result2 is False, \
            f'expected False for missing id, got {result2}'
        passed += 1; print('\u2705 Check 5: returns False for missing id')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def delete_item(session, item_id):
    item = session.get(Item, item_id)
    if item is None:
        return False
    session.delete(item)
    session.commit()
    return True
```

</details>